In [1]:
import cv2 as cv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Edge Detection
There are several edge detection options using OpenCV

Canny → best overall

Sobel/Scharr → gradient-based

Laplacian → second derivative, detects fine edges

Prewitt → custom kernel

# Canny
--Pre-processing:

cv.GaussianBlur() helps reduce noise → cleaner edges.

--Thresholds:

cv.Canny(img, lower, upper)

Lower threshold detects weak edges.

Upper threshold detects strong edges.

Good starting values: (100, 200) or (50, 150).

In [2]:
import cv2 as cv
cam=cv.VideoCapture(0)
while True:
    _,img=cam.read()
    img=cv.flip(img,1)
    edge=cv.Canny(img,0,100)
    cv.imshow("Frames",img)
    cv.imshow("Edges",edge)
    if(cv.waitKey(10)==27):
        cam.release()
        break
    

In [3]:
import cv2 as cv

cam = cv.VideoCapture(0)

while True:
    _, img = cam.read()
    
    img = cv.flip(img, 1)

    edge1 = cv.Canny(img, 50, 150)
    edge2 = cv.Canny(img, 100, 200)
    edge3 = cv.Canny(img, 150, 250)

    cv.imshow("Frame", img)
    cv.imshow("Edges1", edge1)
    cv.imshow("Edges2", edge2)
    cv.imshow("Edges3", edge3)

    if cv.waitKey(1) & 0xFF == 27:  # ESC key
        break

cam.release()
cv.destroyAllWindows()

#edge1 shows the most noise. As the range increases it is barely able to detect edges. So,it depends on the requirement

# Sobel
Sobel operator in OpenCV, you don’t use cv.Canny, but instead compute gradients with cv.Sobel.

In [4]:
import cv2 as cv

cam = cv.VideoCapture(0)

while True:
    _, img = cam.read()

    img = cv.flip(img, 1)
    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)

    # Sobel edge detection
    sobelx = cv.Sobel(gray, cv.CV_64F, 1, 0, ksize=3)  # X direction
    sobely = cv.Sobel(gray, cv.CV_64F, 0, 1, ksize=3)  # Y direction

    # Convert back to uint8
    abs_sobelx = cv.convertScaleAbs(sobelx)
    abs_sobely = cv.convertScaleAbs(sobely)

    # Combine both directions
    sobel_combined = cv.bitwise_or(abs_sobelx, abs_sobely)

    cv.imshow("Original", img)
    cv.imshow("Sobel X", abs_sobelx)
    cv.imshow("Sobel Y", abs_sobely)
    cv.imshow("Sobel Combined", sobel_combined)

    if cv.waitKey(1) & 0xFF == 27:  # ESC key
        break

cam.release()
cv.destroyAllWindows()


# Laplacian
Laplacian operator, which finds edges based on second-order derivatives

In [5]:
import cv2 as cv

cam = cv.VideoCapture(0)

while True:
    ret, img = cam.read()
    if not ret:
        print("Failed to grab frame")
        break

    img = cv.flip(img, 1)

    # Split into channels
    b, g, r = cv.split(img)

    # Apply Laplacian on each channel
    lap_b = cv.convertScaleAbs(cv.Laplacian(b, cv.CV_64F, ksize=3))
    lap_g = cv.convertScaleAbs(cv.Laplacian(g, cv.CV_64F, ksize=3))
    lap_r = cv.convertScaleAbs(cv.Laplacian(r, cv.CV_64F, ksize=3))

    # Merge back to get colored edges
    lap_colored = cv.merge((lap_b, lap_g, lap_r))

    cv.imshow("Original", img)
    cv.imshow("Colored Laplacian", lap_colored)

    if cv.waitKey(1) & 0xFF == 27:  # ESC key
        break

cam.release()
cv.destroyAllWindows()


# Prewitt 
The Prewitt operator is similar to Sobel, but instead of weighted kernels, it uses simpler kernels to detect horizontal and vertical edges.
OpenCV does not have a direct cv.Prewitt(), but we can implement it using cv.filter2D() with custom kernels.

For RGB images, we’ll apply Prewitt separately on each channel (B, G, R), then merge them back to create a colored edge map.

In [6]:
import cv2 as cv
import numpy as np

cam = cv.VideoCapture(0)

# Define Prewitt kernels
kernelx = np.array([[-1, 0, 1],
                    [-1, 0, 1],
                    [-1, 0, 1]], dtype=np.float32)

kernely = np.array([[-1, -1, -1],
                    [ 0,  0,  0],
                    [ 1,  1,  1]], dtype=np.float32)

while True:
    ret, img = cam.read()
    if not ret:
        print("Failed to grab frame")
        break

    img = cv.flip(img, 1)

    # Split into channels
    b, g, r = cv.split(img)

    # Apply Prewitt on each channel (X and Y directions)
    bx = cv.filter2D(b, -1, kernelx)
    by = cv.filter2D(b, -1, kernely)
    gx = cv.filter2D(g, -1, kernelx)
    gy = cv.filter2D(g, -1, kernely)
    rx = cv.filter2D(r, -1, kernelx)
    ry = cv.filter2D(r, -1, kernely)

    # Combine X and Y gradients per channel
    b_prewitt = cv.addWeighted(bx, 0.5, by, 0.5, 0)
    g_prewitt = cv.addWeighted(gx, 0.5, gy, 0.5, 0)
    r_prewitt = cv.addWeighted(rx, 0.5, ry, 0.5, 0)

    # Merge channels back to get colored Prewitt edges
    prewitt_colored = cv.merge((b_prewitt, g_prewitt, r_prewitt))

    cv.imshow("Original", img)
    cv.imshow("Prewitt RGB Edges", prewitt_colored)

    if cv.waitKey(1) & 0xFF == 27:  # ESC key
        break

cam.release()
cv.destroyAllWindows()
